In [ ]:
import sys, os, subprocess
from pathlib import Path

# Colab: clone repo and install deps. Local: resolve root from CWD.
try:
    import google.colab  # noqa
    REPO = '/content/Katabatic'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO],
            check=True
        )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    ROOT = Path(REPO)
except ImportError:
    ROOT = Path.cwd().resolve()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists() or (ROOT / 'raw_data').exists():
            break
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.tvae.models import TVAEModel

In [ ]:
MODEL = lambda: TVAEModel(
    embedding_dim=128,
    compress_dims=(128, 128),
    decompress_dims=(128, 128),
    l2scale=1e-5,
    batch_size=500,
    epochs=300,
    loss_factor=2,
    cuda=True,
)

In [ ]:
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

In [ ]:
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"TVAE -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    synthetic_dir = str(ROOT / "synthetic" / dataset / "tvae")

    pipeline = TrainTestSplitPipeline(model=MODEL)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=output_dir,
    )
    print(result)